In [ ]:
# Load historical, SSP5 and mode inputs.
import pandas as pd   
df1 = pd.read_excel(r'C:\Users\Lenovo\Desktop\RF_Water\B_Clustered_panel\03_Clustered_panel_1119.xlsx') 
df1 = df1.rename(columns={'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'})
df1.loc[df1['Both'] == 1, ['Conflict', 'Cooperation']] = 1
#df1['Cooperation'] = df1[label].any(axis=1).astype(int)
df2 = pd.read_excel(r'C:\Users\Lenovo\Desktop\RF_Water\E_ssp_predication\ssp1-5_future_data.xlsx', sheet_name='ssp5')  
df2 = df2[df2['year'] == 2050]  
df3 = pd.read_csv(r'C:\Users\Lenovo\Desktop\RF_Water\D_strategy_onehot\20241218_updated_future_data_with_modes.csv')
df3.rename(columns={
    'ccode_iso3': 'country_code',
    'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'
}, inplace=True)
df3 = df3[df3['year'] == 2050]
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative Planning and Adaptation Strategies'] 
df3_labels = df3[['country_code'] + label]  
df2 = pd.merge(df2, df3_labels, on='country_code', how='left')  
#df2.to_excel('D:\\OneDrive\\Desktop\\df2.xlsx', index=False)


In [ ]:
# Append renamed future data and assign risk flags.
import pandas as pd 
df2.rename(columns={  
    'country': 'Country',
    'country_code': 'ccode_iso3',
    'population_density_ssp5': 'population_density',
    'GDP per capita_ssp5(constant 2015 US$)': 'gdp_percapita',
    'rural_population_ssp5(%)': 'rural_population',
    'precipitation_ssp5': 'precipitation',
    'bws_ssp5': 'bws_score'}, inplace=True)
df = pd.concat([df1, df2], ignore_index=True, sort=True)
df4 = pd.read_excel(r'C:\Users\Lenovo\Desktop\RF_Water\B_Clustered_panel\03_Clustered_panel_1119\Clustered_Conflict_Probability_241119.xlsx')
high_conflict_country = df4[df4['Cluster'] == 1]  
high_conflict_country = high_conflict_country['ccode_iso3'].tolist()  
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Conflict'] = 1
#df.to_excel('D:\\OneDrive\\Desktop\\df.xlsx', index=False)


In [ ]:
# Initialize future conflict averages; exponential-smoothing forecast and export.
'''
df_2019_Average_conflict = df[df['year'] == 2019][['ccode_iso3', 'Average_conflict']]  
conflict_dict = df_2019_Average_conflict.set_index('ccode_iso3')['Average_conflict'].to_dict()  
df.loc[df['year'] == 2050, 'Average_conflict'] = df[df['year'] == 2050]['ccode_iso3'].map(conflict_dict)  
'''
df = df[df['ccode_iso3'] == 'LSO']
import pandas as pd  
from statsmodels.tsa.holtwinters import ExponentialSmoothing    
predictions_2050 = pd.DataFrame(columns=['ccode_iso3', 'Average_conflict_2050'])  
for country in high_conflict_country:  
    country_data = df[(df['ccode_iso3'] == country) & (df['year'].between(1951, 2019))]  
    if not country_data.empty:  
        print(f"Data for {country}:")  
        print(country_data[['Average_conflict']])
        model = ExponentialSmoothing(country_data['Average_conflict'], trend='add', seasonal=None, seasonal_periods=1).fit()  
        forecast = model.forecast(31)
        print(f"Forecast for {country} from 2020 to 2050:")  
        print(forecast)
        if not forecast.empty and forecast.iloc[-1] < 0:  
            forecast.iloc[-1] = 0
        if not forecast.empty:  
            forecast_2050 = forecast.iloc[-1]  
            new_prediction = pd.DataFrame({'ccode_iso3': [country], 'Average_conflict_2050': [forecast_2050]})  
            predictions_2050 = pd.concat([predictions_2050, new_prediction], ignore_index=True)  
        else:  
            print(f"No forecast available for {country}")  
df = df.merge(predictions_2050, on='ccode_iso3', how='left')  
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Average_conflict'] = df['Average_conflict_2050']  
df.drop(columns=['Average_conflict_2050'], inplace=True)  
# print(df)  
df.to_excel('D:\\OneDrive\\Desktop\\time_series_LSO.xlsx', index=False)
